# Python Classes

> The core idea of this chapter is to try to build a correct mental model of Python's object model rather than memorizing syntax.

---

## 1. What is a Class?

A **class** is a blueprint that defines a **new type**.

It describes:

- What data objects of this type can hold (**attributes**)
- What they can do (**methods**)

Example:

```python
class Dog:
    species = "Canine"

    def bark(self):
        print("Woof!")
```

Here:

- `species` → class attribute
- `bark()` → instance method

Creating an object:

```python
dog = Dog()
```

`dog` is an **instance** of `Dog`.

---

## 2. Objects Have State and Behavior

Every object consists of:

### State

Stored in attributes.

```python
dog.name = "Rocky"
dog.age = 5
```

### Behavior

Implemented using methods.

```python
dog.bark()
```

---

## 3. What is `self` ?

`self` is simply a reference to the current object.

When you write

```python
dog.bark()
```

Python roughly transforms it into

```python
Dog.bark(dog)
```

So

```python
def bark(self):
```

receives the object automatically.

`self` is **not a keyword**.

It is only a naming convention.

This is valid (but discouraged):

```python
def bark(me):
    print(me)
```

---

## 4. Python Classes Execute at Runtime

Unlike Java or C++, Python does not merely declare classes.

A class definition is executable code.

```python
class A:
    x = 10
```

When Python reaches this statement:

1. Creates a new namespace
2. Executes the class body
3. Creates a class object
4. Assigns the name `A`

Conceptually:

```python
namespace = {}

namespace["x"] = 10

A = type("A", (), namespace)
```

The official documentation describes class creation as executing the class body in a new namespace and then constructing the class with `type(name, bases, namespace)`. :contentReference[oaicite:1]{index=1}

---

## 5. Classes are Objects

Everything in Python is an object.

Classes are no exception.

```python
print(type(int))
print(type(str))
print(type(list))
```

Output

```python
<class 'type'>
<class 'type'>
<class 'type'>
```

Even user-defined classes:

```python
class A:
    pass

print(type(A))
```

Output

```python
<class 'type'>
```

---

## 6. The `type()` Function Has Two Uses

### Inspect a type

```python
type(5)
```

Output

```python
<class 'int'>
```

---

### Create a class

```python
B = type(
    "B",
    (),
    {
        "x": 40
    }
)
```

Equivalent to

```python
class B:
    x = 40
```

Arguments:

```python
type(
    class_name,
    base_classes,
    namespace
)
```

---

## 7. Namespace Dictionary

The third argument of `type()` is simply the namespace of the class.

Example:

```python
B = type(
    "B",
    (),
    {
        "x": 40,
        "y": 50
    }
)
```

Results in

```python
B.x == 40
B.y == 50
```

Methods are just functions stored inside this namespace.

```python
def hello(self):
    print("Hello")

B = type(
    "B",
    (),
    {
        "hello": hello
    }
)
```

---

## 8. Classes Can Be Modified at Runtime

Since classes are objects:

```python
class A:
    pass

A.x = 10
```

Now

```python
print(A.x)
```

prints

```python
10
```

Methods can also be added dynamically.

```python
def greet(self):
    print("Hello")

A.greet = greet
```

---

## 9. Creating a New Class vs Modifying an Existing One

These are different.

### Modifying

```python
A.B.x = 40
```

Adds an attribute to the existing class.

---

### Replacing

```python
A.B = type(
    "B",
    (),
    {
        "x": 40
    }
)
```

Creates a completely new class object.

The old class is not modified.

---

## 10. Nested Classes

Python allows

```python
class Car:

    class Engine:
        pass
```

`Engine` is simply an attribute of `Car`.

It is **not** automatically created for every `Car` object.

Access:

```python
Car.Engine
```

---

## 11. Python Scopes

Python does **not** create scopes for

- if
- for
- while
- try
- with

Example

```python
if True:
    x = 10

print(x)
```

prints

```python
10
```

New scopes are created primarily by

- functions
- lambdas
- comprehensions
- generators

This differs from JavaScript where blocks create scope with `let` and `const`.

---

## 12. Instance vs Class Attributes

```python
class Dog:
    species = "Canine"
```

Every instance can access

```python
dog.species
```

If an instance defines

```python
dog.species = "Wolf"
```

then only that instance changes.

The class attribute remains

```python
Dog.species
```

---

## 13. Instance Attribute Lookup

Python searches attributes in this order:

```
Instance

↓

Class

↓

Base Classes (following MRO)

↓

object
```

The search stops immediately when found.

---

## 14. Inheritance

```python
class Animal:

    def speak(self):
        print("Animal")

class Dog(Animal):
    pass
```

`Dog` inherits everything from `Animal`.

---

## 15. Method Resolution Order (MRO)

Definition:

> MRO (Method Resolution Order) is the order in which Python searches classes when looking for attributes or methods.

Example

```python
class A:
    pass

class B(A):
    pass

class C(B):
    pass
```

MRO

```
C

↓

B

↓

A

↓

object
```

Inspect using

```python
C.__mro__

# or

C.mro()
```

The MRO is computed when the class is created and stored in `__mro__`. :contentReference[oaicite:2]{index=2}

---

## 16. Multiple Inheritance

```python
class Flyer:
    pass

class Swimmer:
    pass

class Duck(Flyer, Swimmer):
    pass
```

Search order

```
Duck

↓

Flyer

↓

Swimmer

↓

object
```

Left-to-right is the intuition for simple cases.

Internally Python uses the **C3 Linearization Algorithm**.

---

## 17. The Diamond Problem

```
      A
     / \
    B   C
     \ /
      D
```

Without MRO

Python could visit

```
A

twice
```

Python's C3 algorithm guarantees

- every class appears once
- left-to-right ordering is preserved
- monotonic ordering
- consistent lookup

If Python cannot produce a consistent order, class creation fails with `TypeError`. :contentReference[oaicite:3]{index=3}

---

## 18. super()

The biggest misconception:

**Incorrect**

> super() calls the parent class.

**Correct**

> super() calls the next class in the Method Resolution Order.

Example

```python
class A:

    def hello(self):
        print("A")

class B(A):

    def hello(self):
        print("B")
        super().hello()
```

Single inheritance

```
B

↓

A
```

Looks like "call parent."

---

Multiple inheritance

```
D

↓

B

↓

C

↓

A
```

Inside `B`

```python
super().hello()
```

calls

```
C
```

NOT

```
A
```

because `C` is next in the MRO.

This is why `super()` is dynamic. It follows the MRO of the **actual instance**, not simply the declared parent. :contentReference[oaicite:4]{index=4}

---

## 19. Does Python Follow the MRO Automatically?

Yes.

You never manually traverse the MRO.

Simply write

```python
super().method()
```

Python determines

- current class
- current instance
- current MRO

and automatically calls the next implementation.

---

## 20. Mental Model of Attribute Lookup

Whenever Python sees

```python
obj.some_attribute
```

it performs

```
Check object

↓

Check class

↓

Walk MRO

↓

Stop immediately once found

↓

Raise AttributeError if never found
```

The MRO is used for **all attribute lookups**, not just methods.

---

## 21. Common Gotchas

### Gotcha 1

```python
class A:

    def hello(name):
        self.name = name
```

Wrong.

Should be

```python
def hello(self, name):
```

---

### Gotcha 2

```python
c = A()

print(c.name)
```

Fails unless

```python
self.name
```

was previously assigned.

Python does **not** create attributes automatically.

---

### Gotcha 3

This

```python
A.B = type(...)
```

does **not** modify the existing class.

It creates a new class and reassigns `A.B`.

---

### Gotcha 4

Methods are just attributes.

```python
A.hello = hello
```

is perfectly valid.

---

### Gotcha 5

Everything after the first successful lookup is ignored.

If

```
Instance

↓

Class

↓

Parent
```

finds the attribute in the class,

parents are never searched.

---

## Interview Notes

- ### What is a class?
    A blueprint that defines a new type.

- ### What is an object?
    An instance of a class.

- ### What is self?
    A reference to the current object.

- ### What is type()?
    - One argument → inspect type
    - Three arguments → create a class


### What is MRO?

The order Python searches classes for attributes and methods.

- ### What does super() do?

    Calls the **next class in the Method Resolution Order**, not necessarily the immediate parent.

- ### How do you inspect the MRO?

    ```python
    Class.__mro__

    # or

    Class.mro()
    ```

    ---

- ### Difference between class and instance attributes

Class attributes belong to the class.

Instance attributes belong to each object individually.

---

## Mental Models Worth Remembering

### Class

> A class is an object that acts as a blueprint for creating other objects.

---

### Instance

> A collection of data with behavior defined by its class.

---

### Namespace

> A mapping (conceptually like a dictionary) from names to objects.

---

### MRO

> A precomputed search path that Python follows for every attribute lookup.

---

### super()

> "Call the next implementation in the MRO."

Never think

> "Call my parent."

Think

> "Continue the lookup chain."

That mental model scales from single inheritance to complex multiple inheritance without changing.